# C2.5 · Supply-chain research

**Function C — Red Teaming and Security Research with AI → Security Research with AI**  ·  *Security of AI*

Builds on **[C2.4 · Data-layer research](https://spbreed.github.io/cyber-commons/lessons/C2.4.html)**.

| | |
|---|---|
| Tools used | Sigstore, in-toto, OWASP AIBOM |

## What this lesson is

**What it covers.** Sign a model artefact with Sigstore and detect a tampered adapter.

**Why a security engineer needs it.** Adapter and LoRA provenance, registry tampering, dependency confusion in agent ecosystems. The control it builds is: verify provenance; sign and attest artefacts.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

A model, a dataset, an adapter and an MCP server all arrive the way any dependency arrives — from someone else, usually unsigned, usually pinned to a tag that can move. Provenance questions produce real answers only when they are specific.

> **At CyberTravels.** The third-party MCP server, the model, the OCR library and the adapter all arrived from somebody else, usually unsigned, usually pinned to a tag that can move. R4.

## 2 · The framework

```
   what arrives from someone else

   model weights   signed? by whom? pinned to a digest or a tag?
   dataset         provenance? licence? contaminated with your eval?
   adapter         who built it, against which base, verified how?
   MCP server      whose process? whose tool descriptions in your context?

   a moving tag is not a pin
```

Supply-chain research for AI systems is the ordinary software problem plus two
artefacts that have no mature process at all.

The ordinary part transfers directly: typosquatting, unsigned packages, new
packages with no soak time. The signals that predict a bad dependency have not
changed.

The two new artefacts:

- **Model weights.** Sigstore and in-toto attestation are technically possible
  and rare in practice. There is no download-count equivalent — "popular
  checkpoint" is not provenance, and a fine-tune of a fine-tune has a lineage
  nobody records.
- **Prompt and tool packages.** MCP servers, agent skill bundles, prompt
  libraries. These run *inside* your agent with your agent's authority, and
  there is no signing convention for them at all.

The honest output of this lesson includes stating where no answer currently
exists, because a risk assessment that invents one is worse than a gap.

## 3 · The procedure, as a skill

The same connector is a review when a person loads it and a block when an agent holding production credentials does. The skill scores the ordinary signals first, then re-scores weighted by the authority the artefact will execute with.

In [ ]:
# skills/research/agent-supply-chain-assessment/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: agent-supply-chain-assessment
description: >-
  Score new packages and MCP connectors for typosquatting and ordinary
  supply-chain signals, then re-score them weighted by the authority the agent
  that loads them runs with. Use when an agent installs its own dependencies or
  connects to a server somebody added last week.
allowed-tools: Read, Grep, Glob
---

# The same package is a different risk inside an agent

Supply-chain assessment for agents differs in one term: the artefact runs with
the agent's authority. A connector that trips three ordinary signals is a
review; the same connector loaded by an agent holding production credentials is
a block. Weighting by authority is what turns the ordinary assessment into the
right answer.

## When to use this

When an agent can install packages, when an MCP server is added, and at any
review of what a coding agent is allowed to pull.

## Procedure

**1 — Establish the known-good set.** The packages this project actually uses.
Typosquat detection is a comparison against something; without the set it is a
spell-check.

**2 — Score edit distance against known-good names.** A distance of one or two
from a popular package, with a recent first-publication date, is the classic
shape. Report the package it imitates, not just the score.

**3 — Apply the ordinary signals.** Age, maintainer count, download history,
whether it was published after the agent asked for it, install scripts.

**4 — Re-score weighted by agent authority.** What credentials are in the
environment the artefact will execute in, and what the agent can reach. This is
the step that moves a review to a block and it needs no new information.

**5 — Report the two verdicts side by side.** Unweighted and authority-weighted.
The difference is the argument for gating what agents may install, and it is
easier to make with both numbers present.

## Output contract

```json
{
  "known_good": ["str"],
  "artefacts": [{"name": "str", "kind": "package|mcp", "distance": 0, "imitates": "str|null",
                 "signals": ["str"], "verdict": "allow|review|block"}],
  "authority": {"credentials_present": ["str"], "reachable": ["str"]},
  "weighted": [{"name": "str", "verdict": "allow|review|block", "moved": true}]
}
```

## Failure modes

- **Typosquat detection with no known-good set.** Everything is close to
  something.
- **Assessing the package and not the environment.** The authority is the term
  that differs.
- **Treating an MCP connector as configuration.** It is code that runs with the
  agent.
"""

In [ ]:
# Execute the skill above, using the shared runtime rather than a copy.
import glob, importlib.util, os, sys

# Kaggle mounts an attached kernel under /kaggle/input, and it uses two
# different layouts — /kaggle/input/<slug>/ on some kernels and
# /kaggle/input/notebooks/<user>/<slug>/ on others. Both were observed on the
# same account in the same hour, so match either. The recursive glob is cheap
# here because /kaggle/input holds only what is attached; globbing the working
# tree instead cost eleven seconds a notebook.
_WHERE = (sorted(glob.glob("/kaggle/input/**/cyber-commons-skill-runtime/__script__.py",
                           recursive=True))
          + [os.path.join(p, "skills/_runtime/cyber_commons_skill_runtime.py")
             for p in (".", "..", "../..")])
_found = next((p for p in _WHERE if os.path.isfile(p)), None)
if _found is None:
    # Say what was looked for and what is actually there. "The runtime is
    # missing" on its own costs whoever hits it an afternoon.
    raise SystemExit("The shared skill runtime is missing."
                     "  looked at: " + repr(_WHERE) +
                     "  /kaggle/input holds: " +
                     repr(glob.glob("/kaggle/input/**", recursive=True)[:20]) +
                     "  cwd: " + os.getcwd() +
                     ". On Kaggle it is attached to this notebook as a "
                     "source; locally it is skills/_runtime/ in the repository.")
_spec = importlib.util.spec_from_file_location("cyber_commons_skill_runtime", _found)
cyber_commons_skill_runtime = importlib.util.module_from_spec(_spec)
sys.modules["cyber_commons_skill_runtime"] = cyber_commons_skill_runtime
_spec.loader.exec_module(cyber_commons_skill_runtime)
from cyber_commons_skill_runtime import run_skill

# Split skills/research/agent-supply-chain-assessment/SKILL.md into the two halves an agent uses —
# the frontmatter it routes on, and the body it follows.
meta, body = run_skill(SKILL_MD)

In [ ]:
# skills/research/agent-supply-chain-assessment/scripts/agent_supply_chain_assessment.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Score new packages and MCP connectors for typosquatting and then re-score them weighted by the authority the agent runs with.

This is the executable half of the `agent-supply-chain-assessment` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

from dataclasses import dataclass

@dataclass(frozen=True)
class Package:
    name: str; version: str; signed: bool = False
    downloads: int = 0; age_days: int = 999

KNOWN_GOOD = {"requests", "urllib3", "numpy", "pandas", "cryptography",
              "pytest", "flask", "colorama", "langchain"}

def levenshtein(a, b):
    if len(a) < len(b): a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (ca != cb)))
        prev = cur
    return prev[-1]

def typosquat(pkg, known=KNOWN_GOOD):
    if pkg.name in known:
        return None
    near = sorted((levenshtein(pkg.name, k), k) for k in known)[:1]
    if near and near[0][0] <= 2:
        return f"distance {near[0][0]} from popular package {near[0][1]!r}"
    return None

def assess(pkg):
    flags = []
    if not pkg.signed:        flags.append("unsigned — no attestation to source")
    if pkg.age_days < 30:     flags.append(f"published {pkg.age_days}d ago — no soak time")
    if pkg.downloads < 1000:  flags.append(f"only {pkg.downloads} downloads")
    if (t := typosquat(pkg)): flags.append(t)
    verdict = "block" if len(flags) >= 3 else "review" if flags else "allow"
    return verdict, flags

for p in [Package("requests", "2.31.0", True, 900_000, 400),
          Package("requsts", "2.31.0", False, 12, 3),
          Package("colourama", "0.4.6", False, 40, 9),
          Package("langchain", "0.2.1", False, 400_000, 200)]:
    v, flags = assess(p)
    print(f"{p.name+'=='+p.version:24s}{v}")
    for f in flags: print(f"      · {f}")

NEW_ARTEFACTS = {
 "model weights": {
   "signing": "Sigstore/in-toto possible, rarely used",
   "popularity signal": "NONE — 'popular checkpoint' is not provenance",
   "lineage": "a fine-tune of a fine-tune; base model often unrecorded",
   "runs with": "no authority of its own — but shapes every decision",
   "honest verdict": "assess the PUBLISHER, because you cannot assess the artefact"},
 "prompt / tool packages (MCP, skills)": {
   "signing": "NO convention exists",
   "popularity signal": "star counts, which are trivially gamed",
   "lineage": "none recorded",
   "runs with": "YOUR AGENT'S AUTHORITY — this is the dangerous one",
   "honest verdict": "treat as executable code, because it is"},
}
for artefact, props in NEW_ARTEFACTS.items():
    print(f"=== {artefact} ===")
    for k, v in props.items():
        print(f"   {k:20s} {v}")
    print()

# An MCP tool package assessed with the ordinary signals — they still fire.
mcp_pkg = Package("mcp-jira-connector", "0.0.3", signed=False,
                  downloads=180, age_days=6)
v, flags = assess(mcp_pkg)
print(f"{mcp_pkg.name}: {v}")
for f in flags: print(f"   · {f}")
print("\nGood news: the existing process EXTENDS to it rather than needing")
print("invention. Bad news: nothing in that process accounts for the fact that")
print("this package will run with your agent's tools.")

def authority_weighted(pkg, runs_with_agent_authority, agent_blast):
    v, flags = assess(pkg)
    if runs_with_agent_authority and v != "allow":
        return "block", flags + [f"runs with agent authority (blast {agent_blast})"]
    return v, flags

v2, flags2 = authority_weighted(mcp_pkg, True, agent_blast=43)
print(f"\nauthority-weighted verdict: {v2}")
for f in flags2: print(f"   · {f}")
assert v2 == "block"

def risk_assessment(artefact, signals_available):
    known = [s for s, ok in signals_available.items() if ok]
    unknown = [s for s, ok in signals_available.items() if not ok]
    return {
      "artefact": artefact,
      "assessed_on": known,
      "cannot_assess": unknown,
      "statement": (f"assessed on {len(known)}/{len(signals_available)} signals; "
                    f"{', '.join(unknown)} not available for this artefact class"),
    }

for artefact, sig in (
  ("python package", {"signature": True, "downloads": True, "age": True, "lineage": True}),
  ("model weights",  {"signature": False, "downloads": False, "age": True, "lineage": False}),
  ("MCP tool pack",  {"signature": False, "downloads": False, "age": True, "lineage": False}),
):
    r = risk_assessment(artefact, sig)
    print(f"{r['artefact']:18s}{r['statement']}")
print("\nThat last sentence is the deliverable. A risk rating that hides which")
print("signals were unavailable is a number someone will later rely on.")

## What you just proved

The two legitimate packages are allowed or reviewed; both typosquats are blocked with the distance and the package they imitate. The MCP connector trips three ordinary signals and is escalated to block once agent authority is weighted in. The final assessments state explicitly which signals are unavailable for model weights and tool packages.

## Your turn

Add one question to your third-party assessment: "does this artefact execute with our agent's authority?" Anything answering yes should not be assessed on the same scale as a library.

---

**Next → [C2.6 · Benchmarks, reproducibility and the research harness](https://spbreed.github.io/cyber-commons/lessons/C2.6.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C2.5.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C2.5.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*